# Course Q&A RAG System

A Retrieval-Augmented Generation (RAG) system that answers student questions from the materials of three courses, stored in different file formats (TXT / DOCX / CSV).

**Pipeline:**
1. Load course chunks from the source files (`.txt`, `.docx`, `.csv`)
2. Hybrid retrieval: BM25 + dense embeddings (`bge-small-en-v1.5`), fused with weighted Reciprocal Rank Fusion (RRF)
3. Arabic questions are translated to English before retrieval
4. An LLM (via Groq) answers using only the retrieved passages, in the language of the question, with source attribution

**Courses:** Multimodal Deep Learning · Python 3 Tutorial · Computer Vision (Szeliski)

## 1. Setup

Install dependencies, import libraries, and configure models. The API key is read from the environment (or a `.env` file) and is never stored in the notebook.

In [1]:
import os, re, csv, json, getpass
import chromadb
from docx import Document
from groq import Groq
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv()  
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")

EMBED_MODEL = "BAAI/bge-small-en-v1.5"
LLM_MODEL = "openai/gpt-oss-120b"   
DATA_DIR = "data/raw_formats"

## 2. Load Course Materials

Each course is stored in a different file format. One loader per format returns chunks in the same schema: `id`, `course`, `source`, `section`, `pages`, `text`. A single `load_file()` function picks the right loader from the file extension.

In [2]:
COURSES = {
    "python": ("Python 3 Tutorial", "python_tutorial.txt"),
    "mdl": ("Multimodal Deep Learning", "mdl_course.docx"),
    "cv": ("Computer Vision (Szeliski)", "cv_course.csv"),
}
HEAD = re.compile(r"CHUNK \| id=(.*?) \| section=(.*?) \| pages=(.*)$")


def _mk(cid, course, source, section, pages, text):
    return {"id": cid, "course": course, "source": source,
            "section": section or "", "pages": pages or "", "text": text.strip()}


def load_txt(path, course):
    chunks, cur, buf = [], None, []

    def flush():
        if cur:
            chunks.append(_mk(cur[0], course, os.path.basename(path), cur[1], cur[2], "\n".join(buf)))

    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("### CHUNK | "):
                flush()
                cur = HEAD.match(line[len("### "):]).groups()
                buf = []
            else:
                buf.append(line)
    flush()
    return chunks


def load_docx(path, course):
    chunks, cur, buf = [], None, []

    def flush():
        if cur:
            chunks.append(_mk(cur[0], course, os.path.basename(path), cur[1], cur[2], "\n".join(buf)))

    for p in Document(path).paragraphs:
        if p.style.name == "Heading 2" and p.text.startswith("CHUNK | "):
            flush()
            cur = HEAD.match(p.text).groups()
            buf = []
        elif cur is not None:
            buf.append(p.text)
    flush()
    return chunks


def load_csv(path, course):
    with open(path, encoding="utf-8-sig", newline="") as f:
        return [_mk(r["id"], course, os.path.basename(path), r["section"], r["pages"], r["text"])
                for r in csv.DictReader(f)]


LOADERS = {".txt": load_txt, ".docx": load_docx, ".csv": load_csv}


def load_file(path, course):
    return LOADERS[os.path.splitext(path)[1].lower()](path, course)


chunks = []
for key, (course, fname) in COURSES.items():
    chunks.extend(load_file(os.path.join(DATA_DIR, fname), course))
id2idx = {c["id"]: i for i, c in enumerate(chunks)}

print("total chunks:", len(chunks))
for _, (course, _) in COURSES.items():
    print(f"  {course}: {sum(c['course'] == course for c in chunks)}")

total chunks: 1613
  Python 3 Tutorial: 307
  Multimodal Deep Learning: 510
  Computer Vision (Szeliski): 796


## 3. Build the Index

Two indexes are built over the same chunks:
- **BM25** (lexical): good at rare keywords such as function names and technical terms. It uses a lowercase regex tokenizer that keeps tokens like `split` and `os_path` intact.
- **Dense** (semantic): `bge-small-en-v1.5` embeddings stored in a persistent ChromaDB collection (cosine distance).

If the collection already exists with the right number of chunks, it is reused instead of being rebuilt.

In [3]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ---------- BM25 ----------
TOKEN_RE = re.compile(r"[a-z0-9_]+")


def tokenize(text):
    return TOKEN_RE.findall(text.lower())


bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])

# ---------- Dense + ChromaDB ----------
embedder = SentenceTransformer(EMBED_MODEL, device=device)
db = chromadb.PersistentClient(path="data/chroma")
COLLECTION = "courses_v2"

col = db.get_or_create_collection(COLLECTION, metadata={"hnsw:space": "cosine"})

if col.count() != len(chunks):
    db.delete_collection(COLLECTION)
    col = db.create_collection(COLLECTION, metadata={"hnsw:space": "cosine"})
    emb = embedder.encode([c["text"] for c in chunks], batch_size=32,
                          normalize_embeddings=True, show_progress_bar=True)
    B = 500
    for i in range(0, len(chunks), B):
        b = chunks[i:i + B]
        col.add(
            ids=[c["id"] for c in b],
            embeddings=emb[i:i + B].tolist(),
            documents=[c["text"] for c in b],
            metadatas=[{"course": c["course"], "source": c["source"],
                        "section": c["section"], "pages": str(c["pages"])} for c in b],
        )

print("BM25 docs:", len(chunks), "| Chroma stored:", col.count())

Device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BM25 docs: 1613 | Chroma stored: 1613


## 4. Hybrid Retrieval

Each query is run through both indexes, and the two ranked lists are merged with **weighted Reciprocal Rank Fusion (RRF)**:

`score(chunk) = Σ weight / (k + rank)`

BM25 gets twice the weight of the dense retriever (2.0 vs 1.0). In testing, dense search alone missed chunks containing rare keywords (e.g. "Canny"), while BM25 found them. Giving BM25 more weight keeps those chunks in the top results.

Note: bge models expect an instruction prefix on **queries only**, not on documents.

In [4]:
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "


def dense_ids(q, k=30):
    qe = embedder.encode([QUERY_PREFIX + q], normalize_embeddings=True).tolist()
    r = col.query(query_embeddings=qe, n_results=k)
    return r["ids"][0]


def bm25_ids(q, k=30):
    scores = bm25.get_scores(tokenize(q))
    top = sorted(range(len(chunks)), key=lambda i: scores[i], reverse=True)[:k]
    return [chunks[i]["id"] for i in top]


def rrf_w(rank_lists, weights, k=60):
    s = {}
    for lst, w in zip(rank_lists, weights):
        for rank, cid in enumerate(lst, start=1):
            s[cid] = s.get(cid, 0) + w / (k + rank)
    return sorted(s.items(), key=lambda x: x[1], reverse=True)


def search(q, top_k=8, pool=30, w_dense=1.0, w_bm25=2.0):
    fused = rrf_w([dense_ids(q, pool), bm25_ids(q, pool)], [w_dense, w_bm25])
    out = []
    for cid, sc in fused[:top_k]:
        c = chunks[id2idx[cid]]
        out.append((cid, c["course"], round(sc, 4), c["text"][:100].replace("\n", " ")))
    return out


for q in ["How does the split() method work in Python?",
          "What is Difference of Gaussian filter used for in edge detection?",
          "What is contrastive learning in multimodal models?"]:
    print("\nQ:", q)
    for row in search(q, top_k=3):
        print("  ", row)


Q: How does the split() method work in Python?
   ('py-0143', 'Python 3 Tutorial', 0.0492, 'String split() Method Description The split() method returns a list of all the words in the string, ')
   ('py-0155', 'Python 3 Tutorial', 0.0464, "After deleting value at index 2 :  ['physics', 'chemistry', 2000] Note: remove() method is discussed")
   ('py-0173', 'Python 3 Tutorial', 0.0432, "Repetition 3 in (1, 2, 3) True Membership for x in (1,2,3) : print (x, end=' ') 1 2 3 Iteration Inde")

Q: What is Difference of Gaussian filter used for in edge detection?
   ('cv-0720', 'Computer Vision (Szeliski)', 0.0487, '(7.24) (Wiejak, Buxton, and Buxton 1985), which allows for a much more efficient implementation usin')
   ('cv-0717', 'Computer Vision (Szeliski)', 0.0479, 'map). A mathematical way to define the slope and direction of a surface is through its gradient, J(x')
   ('cv-0718', 'Computer Vision (Szeliski)', 0.046, 'review alternative edge detection algorithms and compare their performa

## 5. Answer Generation

The pipeline for each question:
1. **Language detection**: Arabic questions are translated to English (the course materials are in English), so retrieval always runs on an English query.
2. **Retrieval**: the top 8 chunks from the hybrid search.
3. **Generation**: the LLM answers using only those passages, in the same language as the question, and cites passage numbers like `[1]`.
4. **Source attribution**: the system appends the list of retrieved sources (course, section, pages, chunk id).

If the passages do not contain the answer, the model is instructed to say so instead of guessing.

The LLM can be swapped by changing `LLM_MODEL` in the setup cell.

In [6]:
client = Groq()


def llm(messages, temperature=0.2, max_tokens=1024):
    r = client.chat.completions.create(
        model=LLM_MODEL, messages=messages,
        temperature=temperature, max_tokens=max_tokens,
    )
    return r.choices[0].message.content.strip()


def translate_to_en(q):
    return llm([
        {"role": "system", "content": "Translate the user's question to English. If it is already English, return it unchanged. Keep technical terms as they are. Output only the translation."},
        {"role": "user", "content": q},
    ], temperature=0, max_tokens=500)


def is_arabic(text):
    ar = len(re.findall(r"[\u0600-\u06FF]", text))
    return ar > 0.3 * max(1, len(re.findall(r"[A-Za-z\u0600-\u06FF]", text)))


SYSTEM = """You are a Q&A assistant over three books: Multimodal Deep Learning, Python 3 Tutorial, and Computer Vision (Szeliski).
Rules:
- Answer ONLY from the provided passages. Never add outside knowledge.
- Some passages may be irrelevant to the question. Ignore them, and use only the ones that actually answer it.
- If NONE of the passages answer the question, say so clearly and do not guess.
- Cite a passage number like [1] only if that exact passage contains the fact you are stating. Never cite a passage you did not use.
- Do NOT write a "Sources" section or list; the system adds it automatically.
- Be concise and well organized.
- {lang_rule}"""

LANG_AR = ("اكتب الإجابة بالعامية المصرية بس، ممنوع الفصحى. "
           "مثال على الأسلوب: 'ده بيعمل كذا عشان يوصل لكذا، ويعني إن الـ model بيتعلم ازاي...'. "
           "سيب المصطلحات التقنية بالإنجليزي. "
           "لو الإجابة مش في المقاطع، قول بالنص: مش موجود في المصادر اللي معايا.")
LANG_EN = "Reply in English. If the answer is not in the passages, say exactly: Not found in the provided sources."


def answer(question, top_k=8):
    ar = is_arabic(question)
    q_en = translate_to_en(question) if ar else question
    hits = search(q_en, top_k=top_k)
    ids = [h[0] for h in hits]

    ctx, srcs = [], []
    for n, cid in enumerate(ids, 1):
        c = chunks[id2idx[cid]]
        label = f"{c['course']} | {c['section'] or '-'} | p.{c['pages']}"
        ctx.append(f"[{n}] ({label})\n{c['text']}")
        srcs.append(f"[{n}] {label} ({cid})")

    user = "Passages:\n\n" + "\n\n".join(ctx) + f"\n\nQuestion: {question}"
    system = SYSTEM.format(lang_rule=LANG_AR if ar else LANG_EN)
    ans = llm([{"role": "system", "content": system},
               {"role": "user", "content": user}])
    return q_en, ans, srcs


for q in ["How does the split() method work in Python?",
          "إيه هو الـ attention في الـ transformers؟"]:
    q_en, ans, srcs = answer(q)
    print("\n" + "=" * 60)
    print("Q:", q, "\nEN:", q_en, "\n")
    print(ans)
    print("\nSources:")
    print("\n".join(srcs))


Q: How does the split() method work in Python? 
EN: How does the split() method work in Python? 

The `split()` method divides a string into a list of substrings.  
It uses the first argument as the delimiter (default `""` → any whitespace) and can optionally limit the number of splits with the second argument `num`. The call returns a list containing the resulting pieces.  

**Syntax**  
```python
str.split(sep="", maxsplit=string.count(sep))
```  
* `sep` – delimiter string (space by default).  
* `maxsplit` – maximum number of splits to perform.  

Example:  

```python
s = "this is string example....wow!!!"
print(s.split())          # ['this', 'is', 'string', 'example....wow!!!']
print(s.split('i', 1))    # ['th', 's is string example....wow!!!']
print(s.split('w'))       # ['this is string example....', 'o', '!!!']
```  

[1]

Sources:
[1] Python 3 Tutorial | 10. Strings > String split() Method | p.134 (py-0143)
[2] Python 3 Tutorial | 11. Lists | p.144-145 (py-0155)
[3] Python 3

## 6. Evaluation

A small sanity check with questions from each course, in both languages, plus one question whose answer is **not** in the materials. For that one, the system should say it is not found instead of guessing.

In [7]:
TESTS = [
    ("MDL (EN)", "What is contrastive learning in multimodal models?"),
    ("Python (AR)", "ازاي أعمل loop على list في بايثون؟"),
    ("CV (EN)", "What is Difference of Gaussian filter used for in edge detection?"),
    ("Out of scope (EN)", "How do I use a list comprehension in Python?"),
    ("Out of scope (AR)", "إيه هو الـ Docker وازاي بيشتغل؟"),
]

for tag, q in TESTS:
    q_en, ans, srcs = answer(q)
    print("=" * 60)
    print(f"[{tag}] {q}\n")
    print(ans)
    print("\nTop source:", srcs[0])

[MDL (EN)] What is contrastive learning in multimodal models?

Contrastive learning is a training paradigm used to align the representations of different modalities.  
In multimodal settings it works by **pairing semantically equivalent samples from two (or more) modalities and encouraging their embeddings to be close together, while pushing apart embeddings of non‑matching pairs**.  By rotating and scaling the separate modality‑specific spaces through this loss, the model learns coordinated (or joint) representations that can be compared and queried across modalities, enabling tasks such as zero‑shot cross‑modal retrieval【1】.

Top source: [1] Multimodal Deep Learning | 4.1 Including Further Modalities | p.186 (mdl-0391)
[Python (AR)] ازاي أعمل loop على list في بايثون؟

عشان تكرّر (loop) على عناصر الـ list في بايثون، أبسط طريقة هي إنك تستخدم `for` loop وتعرّف متغيّر يمثل كل عنصر في الـ list. مثال من المصدر بيوريك إزاي بنعمل ده مع لستة أرقام:

```python
numbers = [11, 33, 55, 39, 75]
fo

## 7. Web Interface (Gradio)

A simple UI: the user types a question (Arabic or English) and gets the answer plus the list of sources. The same interface is what the Docker image will serve.

In [8]:
import gradio as gr


def ask(question):
    if not question or not question.strip():
        return "Please type a question first.", ""
    _, ans, srcs = answer(question)
    return ans, "\n".join(srcs)


demo = gr.Interface(
    fn=ask,
    inputs=gr.Textbox(label="Question / السؤال", lines=2,
                      placeholder="Ask in Arabic or English..."),
    outputs=[
        gr.Markdown(label="Answer / الإجابة"),
        gr.Textbox(label="Sources / المصادر", lines=9),
    ],
    examples=[
        "How does the split() method work in Python?",
        "إيه هو الـ attention في الـ transformers؟",
        "What is Difference of Gaussian filter used for in edge detection?",
    ],
    title="Course Q&A RAG",
    description="Ask about Multimodal Deep Learning, Python 3, or Computer Vision. Answers come from the course materials only.",
    flagging_mode="never",
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
